In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")
tags = pd.read_csv("tags.csv")
links = pd.read_csv("links.csv")

In [3]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
movies.shape

(9742, 3)

In [5]:
movies.isnull().sum()

movieId    0
title      0
genres     0
dtype: int64

In [6]:
movies["genres"] = movies["genres"].replace("(no genres listed)", "")
movies["genres"] = movies["genres"].str.replace("|", " ", regex=False)

In [7]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy Romance
3,4,Waiting to Exhale (1995),Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy


In [8]:
cv = CountVectorizer()

genre_matrix = cv.fit_transform(movies["genres"])

In [9]:
genre_matrix.shape

(9742, 21)

In [10]:
cv.get_feature_names_out()

array(['action', 'adventure', 'animation', 'children', 'comedy', 'crime',
       'documentary', 'drama', 'fantasy', 'fi', 'film', 'horror', 'imax',
       'musical', 'mystery', 'noir', 'romance', 'sci', 'thriller', 'war',
       'western'], dtype=object)

In [11]:
similarity = cosine_similarity(genre_matrix)

In [12]:
similarity.shape

(9742, 9742)

In [13]:
similarity[0]

array([1.        , 0.77459667, 0.31622777, ..., 0.        , 0.31622777,
       0.4472136 ], shape=(9742,))

In [14]:
movie_indices = pd.Series(movies.index, index=movies["title"])

In [15]:
movie_indices["Toy Story (1995)"]

np.int64(0)

In [16]:
movie_indices["Jumanji (1995)"]

np.int64(1)

In [17]:
def recommend_movies(movie_title, top_n=10):
    if movie_title not in movie_indices:
        return "Movie not found in dataset."

    movie_index = movie_indices[movie_title]

    similarity_scores = list(enumerate(similarity[movie_index]))

    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    similarity_scores = similarity_scores[1:top_n + 1]

    movie_indexes = [i[0] for i in similarity_scores]
    scores = [i[1] for i in similarity_scores]

    recommendations = movies.iloc[movie_indexes][["title", "genres"]].copy()
    recommendations["similarity_score"] = scores

    return recommendations

In [18]:
recommend_movies("Toy Story (1995)", 10)

,title,genres,similarity_score
1706,Antz (1998),Adventure Animation Children Comedy Fantasy,1.0
2355,Toy Story 2 (1999),Adventure Animation Children Comedy Fantasy,1.0
2809,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure Animation Children Comedy Fantasy,1.0
3000,"Emperor's New Groove, The (2000)",Adventure Animation Children Comedy Fantasy,1.0
3568,"Monsters, Inc. (2001)",Adventure Animation Children Comedy Fantasy,1.0
6194,"Wild, The (2006)",Adventure Animation Children Comedy Fantasy,1.0
6486,Shrek the Third (2007),Adventure Animation Children Comedy Fantasy,1.0
6948,"Tale of Despereaux, The (2008)",Adventure Animation Children Comedy Fantasy,1.0
7760,Asterix and the Vikings (Astérix et les Viking...,Adventure Animation Children Comedy Fantasy,1.0
8219,Turbo (2013),Adventure Animation Children Comedy Fantasy,1.0


In [19]:
def recommend_movies(movie_title, top_n=10):
    if movie_title not in movie_indices:
        return "Movie not found in dataset."

    movie_index = movie_indices[movie_title]

    similarity_scores = list(enumerate(similarity[movie_index]))

    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    similarity_scores = similarity_scores[1:top_n + 1]

    movie_indexes = [i[0] for i in similarity_scores]
    scores = [round(i[1], 2) for i in similarity_scores]

    recommendations = movies.iloc[movie_indexes][["title", "genres"]].copy()
    recommendations["similarity_score"] = scores

    return recommendations.reset_index(drop=True)

In [20]:
recommend_movies("Toy Story (1995)", 10)

,title,genres,similarity_score
0,Antz (1998),Adventure Animation Children Comedy Fantasy,1.0
1,Toy Story 2 (1999),Adventure Animation Children Comedy Fantasy,1.0
2,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure Animation Children Comedy Fantasy,1.0
3,"Emperor's New Groove, The (2000)",Adventure Animation Children Comedy Fantasy,1.0
4,"Monsters, Inc. (2001)",Adventure Animation Children Comedy Fantasy,1.0
5,"Wild, The (2006)",Adventure Animation Children Comedy Fantasy,1.0
6,Shrek the Third (2007),Adventure Animation Children Comedy Fantasy,1.0
7,"Tale of Despereaux, The (2008)",Adventure Animation Children Comedy Fantasy,1.0
8,Asterix and the Vikings (Astérix et les Viking...,Adventure Animation Children Comedy Fantasy,1.0
9,Turbo (2013),Adventure Animation Children Comedy Fantasy,1.0


In [21]:
recommend_movies("Jumanji (1995)", 10)

,title,genres,similarity_score
0,"Indian in the Cupboard, The (1995)",Adventure Children Fantasy,1.0
1,"NeverEnding Story III, The (1994)",Adventure Children Fantasy,1.0
2,Escape to Witch Mountain (1975),Adventure Children Fantasy,1.0
3,Darby O'Gill and the Little People (1959),Adventure Children Fantasy,1.0
4,Return to Oz (1985),Adventure Children Fantasy,1.0
5,"NeverEnding Story, The (1984)",Adventure Children Fantasy,1.0
6,"NeverEnding Story II: The Next Chapter, The (1...",Adventure Children Fantasy,1.0
7,Santa Claus: The Movie (1985),Adventure Children Fantasy,1.0
8,Harry Potter and the Sorcerer's Stone (a.k.a. ...,Adventure Children Fantasy,1.0
9,"Chronicles of Narnia: The Lion, the Witch and ...",Adventure Children Fantasy,1.0


In [22]:
recommend_movies("Heat (1995)", 10)

,title,genres,similarity_score
0,Assassins (1995),Action Crime Thriller,1.0
1,Die Hard: With a Vengeance (1995),Action Crime Thriller,1.0
2,"Net, The (1995)",Action Crime Thriller,1.0
3,Natural Born Killers (1994),Action Crime Thriller,1.0
4,Judgment Night (1993),Action Crime Thriller,1.0
5,Batman (1989),Action Crime Thriller,1.0
6,Die Hard (1988),Action Crime Thriller,1.0
7,Hard Rain (1998),Action Crime Thriller,1.0
8,"Replacement Killers, The (1998)",Action Crime Thriller,1.0
9,U.S. Marshals (1998),Action Crime Thriller,1.0


In [23]:
recommend_movies("GoldenEye (1995)", 10)

,title,genres,similarity_score
0,Broken Arrow (1996),Action Adventure Thriller,1.0
1,Cliffhanger (1993),Action Adventure Thriller,1.0
2,Executive Decision (1996),Action Adventure Thriller,1.0
3,Surviving the Game (1994),Action Adventure Thriller,1.0
4,"Rock, The (1996)",Action Adventure Thriller,1.0
5,Chain Reaction (1996),Action Adventure Thriller,1.0
6,Maximum Risk (1996),Action Adventure Thriller,1.0
7,Die Hard 2 (1990),Action Adventure Thriller,1.0
8,Anaconda (1997),Action Adventure Thriller,1.0
9,Con Air (1997),Action Adventure Thriller,1.0


In [24]:
movies[movies["title"].str.contains("Batman", case=False, na=False)]

,movieId,title,genres
126,153,Batman Forever (1995),Action Adventure Comedy Crime
509,592,Batman (1989),Action Crime Thriller
1060,1377,Batman Returns (1992),Action Crime
1174,1562,Batman & Robin (1997),Action Adventure Fantasy Thriller
2418,3213,Batman: Mask of the Phantasm (1993),Animation Children
5463,26152,Batman (1966),Action Adventure Comedy
5620,27155,"Batman/Superman Movie, The (1998)",Action Adventure Animation Children Fantasy Sc...
5631,27311,Batman Beyond: Return of the Joker (2000),Action Animation Crime Sci-Fi Thriller
5917,33794,Batman Begins (2005),Action Crime IMAX
6815,60979,Batman: Gotham Knight (2008),Action Animation Crime


In [25]:
movies[movies["title"].str.contains("Avengers", case=False, na=False)]

,movieId,title,genres
1611,2153,"Avengers, The (1998)",Action Adventure
6148,44020,Ultimate Avengers (2006),Action Animation Children Sci-Fi
7693,89745,"Avengers, The (2012)",Action Adventure Sci-Fi IMAX
8551,115727,Crippled Avengers (Can que) (Return of the 5 D...,Action Adventure
8686,122892,Avengers: Age of Ultron (2015),Action Adventure Sci-Fi
8693,122912,Avengers: Infinity War - Part I (2018),Action Adventure Sci-Fi
9153,147657,Masked Avengers (1981),Action
9488,170297,Ultimate Avengers 2 (2006),Action Animation Sci-Fi


In [26]:
movies["genres"].str.split(" ").explode().unique()

array(['Adventure', 'Animation', 'Children', 'Comedy', 'Fantasy',
       'Romance', 'Drama', 'Action', 'Crime', 'Thriller', 'Horror',
       'Mystery', 'Sci-Fi', 'War', 'Musical', 'Documentary', 'IMAX',
       'Western', 'Film-Noir', ''], dtype=object)

In [27]:
def search_movie(movie_name):
    result = movies[movies["title"].str.contains(movie_name, case=False, na=False)]
    return result[["movieId", "title", "genres"]].reset_index(drop=True)

In [28]:
search_movie("batman")

,movieId,title,genres
0,153,Batman Forever (1995),Action Adventure Comedy Crime
1,592,Batman (1989),Action Crime Thriller
2,1377,Batman Returns (1992),Action Crime
3,1562,Batman & Robin (1997),Action Adventure Fantasy Thriller
4,3213,Batman: Mask of the Phantasm (1993),Animation Children
5,26152,Batman (1966),Action Adventure Comedy
6,27155,"Batman/Superman Movie, The (1998)",Action Adventure Animation Children Fantasy Sc...
7,27311,Batman Beyond: Return of the Joker (2000),Action Animation Crime Sci-Fi Thriller
8,33794,Batman Begins (2005),Action Crime IMAX
9,60979,Batman: Gotham Knight (2008),Action Animation Crime


In [29]:
search_movie("avengers")

,movieId,title,genres
0,2153,"Avengers, The (1998)",Action Adventure
1,44020,Ultimate Avengers (2006),Action Animation Children Sci-Fi
2,89745,"Avengers, The (2012)",Action Adventure Sci-Fi IMAX
3,115727,Crippled Avengers (Can que) (Return of the 5 D...,Action Adventure
4,122892,Avengers: Age of Ultron (2015),Action Adventure Sci-Fi
5,122912,Avengers: Infinity War - Part I (2018),Action Adventure Sci-Fi
6,147657,Masked Avengers (1981),Action
7,170297,Ultimate Avengers 2 (2006),Action Animation Sci-Fi


In [30]:
recommend_movies("Dark Knight, The (2008)", 10)

,title,genres,similarity_score
0,Need for Speed (2014),Action Crime Drama IMAX,1.00
1,"Fast Five (Fast and the Furious 5, The) (2011)",Action Crime Drama Thriller IMAX,0.89
2,Dead Presidents (1995),Action Crime Drama,0.87
3,Bad Company (1995),Action Crime Drama,0.87
4,Faster Pussycat! Kill! Kill! (1965),Action Crime Drama,0.87
5,Menace II Society (1993),Action Crime Drama,0.87
6,"Substitute, The (1996)",Action Crime Drama,0.87
7,Nothing to Lose (1994),Action Crime Drama,0.87
8,"Untouchables, The (1987)",Action Crime Drama,0.87
9,Monument Ave. (1998),Action Crime Drama,0.87


In [31]:
movie_name = input("Enter a movie name: ")

search_results = search_movie(movie_name)

search_results

Enter a movie name:  jumanji


,movieId,title,genres
0,2,Jumanji (1995),Adventure Children Fantasy
1,179401,Jumanji: Welcome to the Jungle (2017),Action Adventure Children


In [33]:
selected_movie = input("Enter exact movie title from search results: ")

recommend_movies(selected_movie, 10)

Enter exact movie title from search results:  Dark Knight, The (2008)


,title,genres,similarity_score
0,Need for Speed (2014),Action Crime Drama IMAX,1.00
1,"Fast Five (Fast and the Furious 5, The) (2011)",Action Crime Drama Thriller IMAX,0.89
2,Dead Presidents (1995),Action Crime Drama,0.87
3,Bad Company (1995),Action Crime Drama,0.87
4,Faster Pussycat! Kill! Kill! (1965),Action Crime Drama,0.87
5,Menace II Society (1993),Action Crime Drama,0.87
6,"Substitute, The (1996)",Action Crime Drama,0.87
7,Nothing to Lose (1994),Action Crime Drama,0.87
8,"Untouchables, The (1987)",Action Crime Drama,0.87
9,Monument Ave. (1998),Action Crime Drama,0.87


In [34]:
def movie_recommender():
    movie_name = input("Enter a movie name to search: ")

    search_results = search_movie(movie_name)

    if search_results.empty:
        print("No movies found. Try another movie name.")
        return

    print("\nMatching Movies:")
    display(search_results)

    selected_movie = input("\nEnter exact movie title from the above list: ")

    recommendations = recommend_movies(selected_movie, 10)

    print("\nRecommended Movies:")
    display(recommendations)

In [35]:
movie_recommender()

Enter a movie name to search:  batman



Matching Movies:


,movieId,title,genres
0,153,Batman Forever (1995),Action Adventure Comedy Crime
1,592,Batman (1989),Action Crime Thriller
2,1377,Batman Returns (1992),Action Crime
3,1562,Batman & Robin (1997),Action Adventure Fantasy Thriller
4,3213,Batman: Mask of the Phantasm (1993),Animation Children
5,26152,Batman (1966),Action Adventure Comedy
6,27155,"Batman/Superman Movie, The (1998)",Action Adventure Animation Children Fantasy Sc...
7,27311,Batman Beyond: Return of the Joker (2000),Action Animation Crime Sci-Fi Thriller
8,33794,Batman Begins (2005),Action Crime IMAX
9,60979,Batman: Gotham Knight (2008),Action Animation Crime



Enter exact movie title from the above list:  Batman 



Recommended Movies:


'Movie not found in dataset.'

In [36]:
movie_recommender()

Enter a movie name to search:  batman



Matching Movies:


,movieId,title,genres
0,153,Batman Forever (1995),Action Adventure Comedy Crime
1,592,Batman (1989),Action Crime Thriller
2,1377,Batman Returns (1992),Action Crime
3,1562,Batman & Robin (1997),Action Adventure Fantasy Thriller
4,3213,Batman: Mask of the Phantasm (1993),Animation Children
5,26152,Batman (1966),Action Adventure Comedy
6,27155,"Batman/Superman Movie, The (1998)",Action Adventure Animation Children Fantasy Sc...
7,27311,Batman Beyond: Return of the Joker (2000),Action Animation Crime Sci-Fi Thriller
8,33794,Batman Begins (2005),Action Crime IMAX
9,60979,Batman: Gotham Knight (2008),Action Animation Crime



Enter exact movie title from the above list:  Batman Begins (2005)



Recommended Movies:


,title,genres,similarity_score
0,"Dark Knight, The (2008)",Action Crime Drama IMAX,0.87
1,Eagle Eye (2008),Action Crime Thriller IMAX,0.87
2,"Dark Knight Rises, The (2012)",Action Adventure Crime IMAX,0.87
3,"Good Day to Die Hard, A (2013)",Action Crime Thriller IMAX,0.87
4,"Fast & Furious 6 (Fast and the Furious 6, The)...",Action Crime Thriller IMAX,0.87
5,Need for Speed (2014),Action Crime Drama IMAX,0.87
6,New York Cop (Nyû Yôku no koppu) (1993),Action Crime,0.82
7,Striking Distance (1993),Action Crime,0.82
8,Set It Off (1996),Action Crime,0.82
9,Batman Returns (1992),Action Crime,0.82


In [37]:
def movie_recommender():
    movie_name = input("Enter a movie name to search: ")

    search_results = search_movie(movie_name)

    if search_results.empty:
        print("No movies found. Try another movie name.")
        return

    print("\nMatching Movies:")
    display(search_results)

    selected_index = int(input("\nEnter index number from the above list: "))

    if selected_index not in search_results.index:
        print("Invalid index number.")
        return

    selected_movie = search_results.loc[selected_index, "title"]

    recommendations = recommend_movies(selected_movie, 10)

    print(f"\nRecommended Movies for: {selected_movie}")
    display(recommendations)

In [38]:
movie_recommender()

Enter a movie name to search:  batman



Matching Movies:


,movieId,title,genres
0,153,Batman Forever (1995),Action Adventure Comedy Crime
1,592,Batman (1989),Action Crime Thriller
2,1377,Batman Returns (1992),Action Crime
3,1562,Batman & Robin (1997),Action Adventure Fantasy Thriller
4,3213,Batman: Mask of the Phantasm (1993),Animation Children
5,26152,Batman (1966),Action Adventure Comedy
6,27155,"Batman/Superman Movie, The (1998)",Action Adventure Animation Children Fantasy Sc...
7,27311,Batman Beyond: Return of the Joker (2000),Action Animation Crime Sci-Fi Thriller
8,33794,Batman Begins (2005),Action Crime IMAX
9,60979,Batman: Gotham Knight (2008),Action Animation Crime



Enter index number from the above list:  8



Recommended Movies for: Batman Begins (2005)


,title,genres,similarity_score
0,"Dark Knight, The (2008)",Action Crime Drama IMAX,0.87
1,Eagle Eye (2008),Action Crime Thriller IMAX,0.87
2,"Dark Knight Rises, The (2012)",Action Adventure Crime IMAX,0.87
3,"Good Day to Die Hard, A (2013)",Action Crime Thriller IMAX,0.87
4,"Fast & Furious 6 (Fast and the Furious 6, The)...",Action Crime Thriller IMAX,0.87
5,Need for Speed (2014),Action Crime Drama IMAX,0.87
6,New York Cop (Nyû Yôku no koppu) (1993),Action Crime,0.82
7,Striking Distance (1993),Action Crime,0.82
8,Set It Off (1996),Action Crime,0.82
9,Batman Returns (1992),Action Crime,0.82
